# Team solution: spatial image–text matching with pairwise ranking

This notebook is designed for the **team phase**. It uses the supplied train and validation labels for model development, then refits the selected architecture on `train.csv + val.csv` and writes `final_submission.csv`.

## Main changes vs. the baseline
- Keeps the spatial grid from a pretrained ImageNet ResNet-18 instead of collapsing the image to one vector before seeing the caption.
- Uses a text-conditioned spatial attention + FiLM-style fusion head.
- Trains with binary cross-entropy plus a pairwise ranking loss. Each `pair_id` has one true and one false caption.
- Fits vocabulary and feature normalization statistics using **train only** during validation experiments. Test images are only passed through the frozen image encoder for inference; they do not determine normalization statistics.
- Selects the epoch using held-out validation ROC-AUC, reports category AUC and pair accuracy, then retrains on train+validation for that number of epochs.

The model is still a dual-encoder-style solution: the image encoder and text encoder process their own modalities separately; the trainable fusion/scoring layers are learned only from the supplied competition data.

## 1. Setup

In [ ]:
import os, random, zipfile, copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import roc_auc_score

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Locate and load the competition data

In [ ]:
# Kaggle: attach the competition dataset under Notebook > Add Input.
# The fallback paths also support running this notebook in a local/Jupyter environment.
candidate_dirs = []
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    candidate_dirs = [p.parent for p in kaggle_input.glob("**/test.csv")
                      if (p.parent / "train.csv").exists()
                      and (p.parent / "val.csv").exists()
                      and (p.parent / "sample_submission.csv").exists()]

if candidate_dirs:
    DATA_DIR = candidate_dirs[0]
    OUTPUT_DIR = Path("/kaggle/working")
else:
    possible_zips = [
        Path("/mnt/data/assessment-of-logical-statements-in-an-image-text-problem.zip"),
        Path("/content/competition_data.zip"),
        Path("./assessment-of-logical-statements-in-an-image-text-problem.zip"),
    ]
    zip_path = next((p for p in possible_zips if p.exists()), None)
    if zip_path is not None:
        DATA_DIR = Path("./competition_data")
        if not (DATA_DIR / "train.csv").exists():
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(DATA_DIR)
        OUTPUT_DIR = Path(".")
    else:
        raise FileNotFoundError(
            "Attach the competition dataset in Kaggle Add Input, or place the supplied ZIP "
            "at /content/competition_data.zip."
        )

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Prefix image paths by split so that equal filenames in different folders cannot collide.
train_df["image_path"] = "train/" + train_df["image"].astype(str)
val_df["image_path"] = "val/" + val_df["image"].astype(str)
test_df["image_path"] = "test/" + test_df["image"].astype(str)

print("Data directory:", DATA_DIR)
print(f"train: {len(train_df):,} captions / {train_df.image.nunique():,} images")
print(f"val:   {len(val_df):,} captions / {val_df.image.nunique():,} images")
print(f"test:  {len(test_df):,} captions / {test_df.image.nunique():,} images")
display(pd.DataFrame({
    "train": train_df["category"].value_counts(),
    "val": val_df["category"].value_counts(),
    "test": test_df["category"].value_counts()
}).fillna(0).astype(int))

## 3. Check pair structure

In [ ]:
def check_pairs(df, split_name):
    grouped = df.groupby("pair_id")["label"].agg(["count", "sum"])
    assert (grouped["count"] == 2).all(), f"{split_name}: every pair_id must have 2 rows"
    assert (grouped["sum"] == 1).all(), f"{split_name}: each pair must have exactly one true caption"
    print(f"{split_name}: {len(grouped):,} pairs checked")

check_pairs(train_df, "train")
check_pairs(val_df, "val")

## 4. Frozen image features

We use the spatial feature map from ImageNet-pretrained ResNet-18. The feature maps are cached once because the backbone is frozen. Normalization statistics are calculated from **train images only**, then applied unchanged to validation and test features.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
feature_extractor = nn.Sequential(*list(backbone.children())[:-2]).to(device).eval()
for p in feature_extractor.parameters():
    p.requires_grad = False

all_image_paths = sorted(set(train_df["image_path"]) | set(val_df["image_path"]) | set(test_df["image_path"]))
image_index = {name: i for i, name in enumerate(all_image_paths)}
train_image_indices = np.array(sorted({image_index[p] for p in train_df["image_path"]}), dtype=np.int64)

BATCH_IMAGES = 96 if device == "cuda" else 32
feature_batches = []
with torch.inference_mode():
    for start in tqdm(range(0, len(all_image_paths), BATCH_IMAGES), desc="Extracting ResNet-18 maps"):
        batch_paths = all_image_paths[start:start + BATCH_IMAGES]
        images = []
        for rel_path in batch_paths:
            with Image.open(DATA_DIR / rel_path) as im:
                images.append(transform(im.convert("RGB")))
        x = torch.stack(images).to(device)
        fmap = feature_extractor(x)
        feature_batches.append(fmap.to(dtype=torch.float16, device="cpu"))

feature_maps_raw = torch.cat(feature_batches, dim=0)  # [N_images, 512, H, W]
# Crucially, only training images determine the feature normalization statistics.
train_maps = feature_maps_raw[torch.as_tensor(train_image_indices)].float()
channel_mean = train_maps.mean(dim=(0, 2, 3), keepdim=True)
channel_std = train_maps.std(dim=(0, 2, 3), keepdim=True).clamp_min(1e-5)
feature_maps = ((feature_maps_raw.float() - channel_mean) / channel_std).half()

print("feature map shape:", tuple(feature_maps.shape))
print("feature statistics fitted on", len(train_image_indices), "training images only")
del feature_batches, feature_maps_raw, train_maps

## 5. Caption vocabulary (fit on train only)

In [ ]:
def tokenize(caption):
    return str(caption).lower().split()

train_tokens = [tokenize(c) for c in train_df["caption"]]
words = sorted({word for toks in train_tokens for word in toks})
vocab = {"<pad>": 0, "<unk>": 1}
for word in words:
    vocab[word] = len(vocab)

# Fixed length is derived only from train captions, with headroom for unseen word orders.
MAX_LEN = max(32, max(len(toks) for toks in train_tokens))
print("vocabulary size:", len(vocab), "| max token length:", MAX_LEN)

def encode_captions(captions):
    ids = np.zeros((len(captions), MAX_LEN), dtype=np.int64)
    lengths = np.zeros(len(captions), dtype=np.int64)
    for i, caption in enumerate(captions):
        toks = tokenize(caption)[:MAX_LEN]
        encoded = [vocab.get(tok, vocab["<unk>"]) for tok in toks]
        ids[i, :len(encoded)] = encoded
        lengths[i] = max(1, len(encoded))
    return torch.from_numpy(ids), torch.from_numpy(lengths)

## 6. Spatial image–text scoring model

The image remains a grid of spatial features. The text encoder produces token-level and pooled features; the caption then conditions spatial attention and feature-wise modulation (FiLM). This lets words such as *left*, *behind*, *red*, and *exactly three* influence how image features are read.

In [ ]:
class SpatialTextMatcher(nn.Module):
    def __init__(self, vocab_size, dim=256, emb_dim=96, gru_hidden=128, grid_h=5, grid_w=8):
        super().__init__()
        self.dim = dim
        self.image_proj = nn.Sequential(
            nn.Conv2d(512, dim, kernel_size=1),
            nn.GroupNorm(16, dim),
            nn.GELU(),
            nn.Conv2d(dim, dim, kernel_size=3, padding=1),
            nn.GroupNorm(16, dim),
            nn.GELU(),
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, grid_h * grid_w, dim))
        nn.init.normal_(self.pos_embed, std=0.02)

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, gru_hidden, batch_first=True, bidirectional=True)
        self.text_attention = nn.Linear(2 * gru_hidden, 1)

        self.query = nn.Linear(dim, dim)
        self.film = nn.Linear(dim, 2 * dim)
        self.classifier = nn.Sequential(
            nn.Linear(5 * dim, 384),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(384, 128),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(128, 1),
        )

    def forward(self, maps, tokens, lengths):
        # Image grid: [B, 512, H, W] -> [B, H*W, dim]
        image_grid = self.image_proj(maps)
        image_tokens = image_grid.flatten(2).transpose(1, 2)
        if image_tokens.shape[1] != self.pos_embed.shape[1]:
            raise ValueError(
                f"Unexpected ResNet grid {tuple(image_grid.shape[-2:])}; "
                "update grid_h/grid_w in SpatialTextMatcher."
            )
        image_tokens = image_tokens + self.pos_embed

        emb = self.embedding(tokens)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        text_seq, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=tokens.shape[1]
        )

        mask = tokens.ne(0)
        text_logits = self.text_attention(text_seq).squeeze(-1)
        text_logits = text_logits.masked_fill(~mask, -1e4)
        text_weights = torch.softmax(text_logits, dim=1)
        text_vec = torch.sum(text_seq * text_weights.unsqueeze(-1), dim=1)

        # Caption-conditioned spatial attention.
        query = self.query(text_vec).unsqueeze(1)
        attn_logits = (image_tokens * query).sum(-1) / (self.dim ** 0.5)
        image_weights = torch.softmax(attn_logits, dim=1)
        attended_image = torch.sum(image_tokens * image_weights.unsqueeze(-1), dim=1)

        # Caption-conditioned feature-wise modulation.
        gamma, beta = self.film(text_vec).chunk(2, dim=-1)
        modulated = image_tokens * (1.0 + torch.tanh(gamma).unsqueeze(1)) + beta.unsqueeze(1)
        modulated_pool = modulated.mean(dim=1)

        fused = torch.cat([
            text_vec,
            attended_image,
            modulated_pool,
            text_vec * attended_image,
            torch.abs(text_vec - attended_image),
        ], dim=-1)
        return self.classifier(fused).squeeze(-1)

## 7. Loaders, pairwise loss and validation

Each training batch contains complete true/false caption pairs. The loss is binary cross-entropy plus a margin-ranking term that encourages the true caption score to exceed the false caption score for the same scene.

In [ ]:
def prepare_rows(df):
    img_idx = torch.tensor(df["image_path"].map(image_index).to_numpy(), dtype=torch.long)
    tokens, lengths = encode_captions(df["caption"].tolist())
    labels = torch.tensor(df["label"].to_numpy(), dtype=torch.float32)
    return img_idx, tokens, lengths, labels

def make_pair_indices(df):
    true_rows, false_rows = [], []
    for _, group in df.groupby("pair_id", sort=False):
        true_idx = group.index[group["label"].to_numpy() == 1].tolist()
        false_idx = group.index[group["label"].to_numpy() == 0].tolist()
        assert len(true_idx) == 1 and len(false_idx) == 1
        true_rows.append(true_idx[0])
        false_rows.append(false_idx[0])
    return torch.tensor(true_rows, dtype=torch.long), torch.tensor(false_rows, dtype=torch.long)

# Reset to positional indices so pair indices index the prepared tensors directly.
train_exp = train_df.reset_index(drop=True).copy()
val_exp = val_df.reset_index(drop=True).copy()
train_img, train_tok, train_len, train_y = prepare_rows(train_exp)
val_img, val_tok, val_len, val_y = prepare_rows(val_exp)
pair_true, pair_false = make_pair_indices(train_exp)

def score_indices(model, row_indices, prepared, batch_size=256):
    img_idx, tokens, lengths, labels = prepared
    outputs = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(row_indices), batch_size):
            ix = row_indices[start:start + batch_size]
            maps = feature_maps[img_idx[ix]].float().to(device)
            logits = model(maps, tokens[ix].to(device), lengths[ix].to(device))
            outputs.append(logits.cpu())
    return torch.cat(outputs).numpy() if outputs else np.array([])

def predict_df(model, df, prepared=None, batch_size=256):
    if prepared is None:
        prepared = prepare_rows(df.reset_index(drop=True))
    return score_indices(model, torch.arange(len(df)), prepared, batch_size=batch_size)

def train_epoch(model, optimizer, pair_true, pair_false, prepared, pair_batch_size=128, pair_weight=0.35, margin=0.5):
    img_idx, tokens, lengths, labels = prepared
    order = torch.randperm(len(pair_true))
    model.train()
    total_loss, seen = 0.0, 0
    bce_fn = nn.BCEWithLogitsLoss()
    for start in range(0, len(order), pair_batch_size):
        sel = order[start:start + pair_batch_size]
        t_idx = pair_true[sel]
        f_idx = pair_false[sel]
        row_idx = torch.cat([t_idx, f_idx])
        maps = feature_maps[img_idx[row_idx]].float().to(device)
        tok = tokens[row_idx].to(device)
        lens = lengths[row_idx].to(device)
        logits = model(maps, tok, lens)
        n = len(t_idx)
        true_logits, false_logits = logits[:n], logits[n:]
        target = torch.cat([
            torch.ones(n, device=device),
            torch.zeros(n, device=device)
        ])
        bce = bce_fn(logits, target)
        ranking = F.relu(margin - (true_logits - false_logits)).mean()
        loss = bce + pair_weight * ranking

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item() * n
        seen += n
    return total_loss / max(seen, 1)

@torch.no_grad()
def evaluate(model, df, prepared):
    logits = predict_df(model, df, prepared=prepared)
    return roc_auc_score(df["label"].to_numpy(), logits), logits

def fit_with_validation(epochs=20, patience=5, lr=8e-4, pair_weight=0.35):
    set_seed()
    model = SpatialTextMatcher(len(vocab)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5
    )
    history = {"train_loss": [], "val_auc": [], "lr": []}
    best_auc, best_epoch, best_state = -1.0, 0, None
    stale = 0
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, optimizer, pair_true, pair_false,
                           (train_img, train_tok, train_len, train_y),
                           pair_weight=pair_weight)
        val_auc, _ = evaluate(model, val_exp, (val_img, val_tok, val_len, val_y))
        scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(loss)
        history["val_auc"].append(val_auc)
        history["lr"].append(current_lr)
        print(f"epoch {epoch:02d} | loss {loss:.4f} | val ROC-AUC {val_auc:.5f} | lr {current_lr:.2e}")

        if val_auc > best_auc:
            best_auc, best_epoch = val_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)
    model.to(device)
    print(f"Best validation ROC-AUC: {best_auc:.5f} at epoch {best_epoch}; elapsed {time.time()-t0:.1f}s")
    return model, history, best_epoch

## 8. Train and inspect validation results

In [ ]:
model, history, best_epoch = fit_with_validation(epochs=20, patience=5, lr=8e-4, pair_weight=0.35)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(history["train_loss"]) + 1), history["train_loss"])
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[1].plot(range(1, len(history["val_auc"]) + 1), history["val_auc"], marker="o")
axes[1].set_title("Validation ROC-AUC")
axes[1].set_xlabel("Epoch")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

val_auc, val_logits = evaluate(model, val_exp, (val_img, val_tok, val_len, val_y))

def pair_accuracy(df, scores):
    tmp = df.reset_index(drop=True).copy()
    tmp["score"] = scores
    correct = []
    for _, g in tmp.groupby("pair_id", sort=False):
        true_score = g.loc[g["label"] == 1, "score"].iloc[0]
        false_score = g.loc[g["label"] == 0, "score"].iloc[0]
        correct.append(float(true_score > false_score) + 0.5 * float(true_score == false_score))
    return float(np.mean(correct))

rows = [{
    "category": "ALL",
    "n": len(val_exp),
    "ROC-AUC": roc_auc_score(val_exp["label"], val_logits),
    "pair_accuracy": pair_accuracy(val_exp, val_logits),
}]
for cat in sorted(val_exp["category"].unique()):
    mask = (val_exp["category"] == cat).to_numpy()
    rows.append({
        "category": cat,
        "n": int(mask.sum()),
        "ROC-AUC": roc_auc_score(val_exp.loc[mask, "label"], val_logits[mask]),
        "pair_accuracy": pair_accuracy(val_exp.loc[mask].reset_index(drop=True), val_logits[mask]),
    })
val_report = pd.DataFrame(rows)
display(val_report.round(4))

## 9. Refit on train + validation

For the team-phase final model, the supplied rules allow using both labeled splits. We keep the architecture and epoch count selected using the held-out validation experiment, then train a fresh model on the combined labeled data. Since `pair_id` values overlap between the train and validation files, the code creates split-specific pair keys before combining them.

In [ ]:
train_final = train_df.copy()
val_final = val_df.copy()
train_final["pair_key"] = "train_" + train_final["pair_id"].astype(str)
val_final["pair_key"] = "val_" + val_final["pair_id"].astype(str)
train_final = train_final.drop(columns=["pair_id"])
val_final = val_final.drop(columns=["pair_id"])
train_final = train_final.rename(columns={"pair_key": "pair_id"})
val_final = val_final.rename(columns={"pair_key": "pair_id"})
final_train_df = pd.concat([train_final, val_final], ignore_index=True)

# Reset index for the paired row-index tensors.
final_train_df = final_train_df.reset_index(drop=True)
final_img, final_tok, final_len, final_y = prepare_rows(final_train_df)
final_true, final_false = make_pair_indices(final_train_df)

set_seed()
final_model = SpatialTextMatcher(len(vocab)).to(device)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=8e-4, weight_decay=1e-4)
for epoch in range(1, best_epoch + 1):
    loss = train_epoch(
        final_model, optimizer, final_true, final_false,
        (final_img, final_tok, final_len, final_y),
        pair_weight=0.35
    )
    print(f"final fit epoch {epoch:02d}/{best_epoch:02d} | loss {loss:.4f}")

## 10. Predict test and write final submission

In [ ]:
test_exp = test_df.reset_index(drop=True).copy()
test_img, test_tok, test_len, test_y_dummy = prepare_rows(
    test_exp.assign(label=0)
)
test_logits = predict_df(
    final_model,
    test_exp.assign(label=0),
    prepared=(test_img, test_tok, test_len, test_y_dummy),
    batch_size=256,
)
test_scores = 1.0 / (1.0 + np.exp(-np.clip(test_logits, -30, 30)))

predictions = pd.DataFrame({"id": test_df["id"].to_numpy(), "label": test_scores})
submission = sample_submission[["id"]].merge(predictions, on="id", how="left", validate="one_to_one")
assert len(submission) == len(sample_submission)
assert submission["label"].notna().all()
assert submission["id"].is_unique
assert submission["label"].between(0, 1).all()

submission_path = OUTPUT_DIR / "final_submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
display(submission.head())

## 11. Experiment record and reproducibility

Record the actual validation metrics from the executed notebook before submission. The final model is trained on train+validation; therefore its own training run does not produce a held-out validation score. Report `val_report` from Section 8 as the validation result for the selected architecture.

- Architecture: frozen ImageNet ResNet-18 spatial maps + BiGRU + text-conditioned attention/FiLM fusion + MLP scorer
- Loss: BCEWithLogitsLoss + 0.35 × pairwise margin ranking loss
- Seed: 42
- Epochs for final fit: `best_epoch` selected by held-out validation ROC-AUC
- Output: `final_submission.csv`

If you want to compare alternatives, change one hyperparameter at a time and keep the validation split fixed. Do not tune against the public leaderboard repeatedly; use it as a limited external check, not as a replacement for validation.